# Stage 2 - Advanced Machine Learning

## Explainable ensemble vehicle-price regression

This is the later and more complete stage of the project. It builds on
the MLC baseline with automated feature selection, Random Forest,
XGBoost, stacking, SHAP, PDP/ICE, linear and non-linear representation
analysis, polynomial regression and exploratory clustering.


## 1. Setup and data split


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from autotrader_price.data import load_vehicle_data, split_features_target
from autotrader_price.evaluation import evaluate_regressor
from autotrader_price.preprocessing import make_preprocessor

RANDOM_STATE = 42
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "adverts.csv"

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30)


from autotrader_price.preprocessing import (
    NUMERICAL,
    make_unsupervised_preprocessor,
)


In [ ]:
data = load_vehicle_data(DATA_PATH)
X_train, X_test, y_train, y_test = split_features_target(
    data,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

print(f"Clean rows: {len(data):,}")
print(f"Training rows: {len(X_train):,}")
print(f"Test rows: {len(X_test):,}")


All learned transformations remain inside fitted pipelines. This avoids
using hold-out information when estimating imputations, encodings or
scaling parameters.


## 2. Automated feature selection


In [ ]:
from sklearn.feature_selection import RFECV, SequentialFeatureSelector
from sklearn.linear_model import LinearRegression

selection_preprocessor = make_preprocessor(scale_output=True)
X_train_selected_space = selection_preprocessor.fit_transform(X_train, y_train)
X_test_selected_space = selection_preprocessor.transform(X_test)

feature_names = (
    selection_preprocessor
    .named_steps["columns"]
    .get_feature_names_out()
)

rfecv = RFECV(
    estimator=LinearRegression(),
    step=1,
    min_features_to_select=4,
    cv=5,
    scoring="r2",
    n_jobs=-1,
)
rfecv.fit(X_train_selected_space, y_train)

rfecv_features = feature_names[rfecv.get_support()]
print("RFECV feature count:", rfecv.n_features_)
print(rfecv_features)


In [ ]:
mean_scores = rfecv.cv_results_["mean_test_score"]
feature_counts = range(
    rfecv.min_features_to_select,
    rfecv.min_features_to_select + len(mean_scores),
)

plt.figure(figsize=(8, 4.5))
plt.plot(feature_counts, mean_scores, marker="o")
plt.axvline(rfecv.n_features_, color="crimson", linestyle="--")
plt.xlabel("Number of selected features")
plt.ylabel("Mean cross-validated R²")
plt.title("Recursive feature elimination with cross-validation")
plt.tight_layout()
plt.show()


In [ ]:
sfs = SequentialFeatureSelector(
    LinearRegression(),
    n_features_to_select="auto",
    direction="forward",
    scoring="r2",
    cv=5,
    n_jobs=-1,
)
sfs.fit(X_train_selected_space, y_train)
sfs_features = feature_names[sfs.get_support()]

linear_sfs = LinearRegression().fit(
    X_train_selected_space[:, sfs.get_support()],
    y_train,
)
sfs_metrics = evaluate_regressor(
    linear_sfs,
    X_test_selected_space[:, sfs.get_support()],
    y_test,
)

print("SFS features:", sfs_features)
print("SFS hold-out metrics:", sfs_metrics)


Feature-selection scores should be interpreted as part of model
development rather than as final unbiased performance estimates. The
untouched hold-out set remains the final comparison.


## 3. Tree ensembles and stacking


In [ ]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

random_forest = Pipeline(
    [
        ("preprocessor", make_preprocessor()),
        (
            "model",
            RandomForestRegressor(
                n_estimators=100,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

xgboost = Pipeline(
    [
        ("preprocessor", make_preprocessor()),
        (
            "model",
            XGBRegressor(
                objective="reg:squarederror",
                n_estimators=100,
                tree_method="hist",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ]
)

stacked_estimator = StackingRegressor(
    estimators=[
        (
            "rf",
            RandomForestRegressor(
                n_estimators=50,
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
        (
            "xgb",
            XGBRegressor(
                objective="reg:squarederror",
                n_estimators=50,
                tree_method="hist",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            ),
        ),
    ],
    final_estimator=LinearRegression(),
    cv=3,
    n_jobs=-1,
)
stacking = Pipeline(
    [
        ("preprocessor", make_preprocessor()),
        ("model", stacked_estimator),
    ]
)

models = {
    "Random Forest": random_forest,
    "XGBoost": xgboost,
    "Stacking": stacking,
}

for name, model in models.items():
    print(f"Fitting {name}...")
    model.fit(X_train, y_train)


## 4. Hold-out comparison


In [ ]:
comparison_rows = []
for name, model in models.items():
    train_metrics = evaluate_regressor(model, X_train, y_train)
    test_metrics = evaluate_regressor(model, X_test, y_test)
    comparison_rows.append(
        {
            "model": name,
            "train_r2": train_metrics["r2"],
            "test_r2": test_metrics["r2"],
            "test_mae": test_metrics["mae"],
            "test_rmse": test_metrics["rmse"],
        }
    )

comparison = (
    pd.DataFrame(comparison_rows)
    .sort_values("test_r2", ascending=False)
    .reset_index(drop=True)
)
comparison


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

score_plot = comparison.melt(
    id_vars="model",
    value_vars=["train_r2", "test_r2"],
    var_name="split",
    value_name="r2",
)
sns.barplot(data=score_plot, x="model", y="r2", hue="split", ax=axes[0])
axes[0].set_title("Train and test R²")
axes[0].set_ylim(0, 1)

sns.barplot(
    data=comparison,
    x="model",
    y="test_mae",
    hue="model",
    legend=False,
    ax=axes[1],
)
axes[1].set_title("Hold-out MAE")

plt.tight_layout()
plt.show()


The original submission recorded test R² values of 0.848 for Random
Forest, 0.861 for XGBoost and 0.862 for stacking. The difference between
XGBoost and stacking was small, so it should be treated as marginal
rather than a decisive improvement.


## 5. Permutation importance


In [ ]:
from sklearn.inspection import permutation_importance

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for axis, name in zip(axes, ["Random Forest", "XGBoost"]):
    result = permutation_importance(
        models[name],
        X_test,
        y_test,
        scoring="r2",
        n_repeats=5,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    importance = pd.Series(
        result.importances_mean,
        index=X_test.columns,
    ).sort_values().tail(8)
    importance.plot(kind="barh", ax=axis)
    axis.set_title(name)
    axis.set_xlabel("Mean decrease in test R²")

plt.suptitle("Permutation importance on original features")
plt.tight_layout()
plt.show()


## 6. SHAP explanations


In [ ]:
import shap

xgb_pipeline = models["XGBoost"]
explain_sample = X_test.sample(min(500, len(X_test)), random_state=RANDOM_STATE)
encoded_sample = xgb_pipeline.named_steps["preprocessor"].transform(explain_sample)
encoded_names = (
    xgb_pipeline
    .named_steps["preprocessor"]
    .get_feature_names_out()
)
encoded_frame = pd.DataFrame(encoded_sample, columns=encoded_names)

xgb_model = xgb_pipeline.named_steps["model"]
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer(encoded_frame)

shap.plots.beeswarm(shap_values, max_display=12)


In [ ]:
shap.plots.waterfall(shap_values[0], max_display=12)


SHAP values explain the fitted model, not the underlying market
mechanism. A large value indicates a strong contribution to a particular
prediction; it does not establish causation.


## 7. Partial dependence and ICE


In [ ]:
from sklearn.inspection import PartialDependenceDisplay

pdp_sample = X_test.sample(min(2_000, len(X_test)), random_state=RANDOM_STATE)
display = PartialDependenceDisplay.from_estimator(
    models["XGBoost"],
    pdp_sample,
    features=["mileage", "year_of_registration"],
    kind="both",
    subsample=300,
    grid_resolution=40,
    random_state=RANDOM_STATE,
    pd_line_kw={"color": "crimson", "linewidth": 2},
    ice_lines_kw={"alpha": 0.08, "color": "steelblue"},
)
display.figure_.suptitle("XGBoost partial dependence and ICE")
plt.tight_layout()
plt.show()


## 8. Scaled PCA and Isomap in model feature space


In [ ]:
from sklearn.decomposition import PCA

representation_preprocessor = make_preprocessor(scale_output=True)
X_representation = representation_preprocessor.fit_transform(X_train, y_train)

pca = PCA(random_state=RANDOM_STATE)
pca.fit(X_representation)
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

plt.figure(figsize=(8, 4.5))
plt.plot(
    np.arange(1, len(cumulative_variance) + 1),
    cumulative_variance,
    marker="o",
)
plt.axhline(0.95, color="crimson", linestyle="--", label="95% threshold")
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA after consistent feature scaling")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.manifold import Isomap

rng = np.random.default_rng(RANDOM_STATE)
sample_size = min(3_000, len(X_representation))
sample_positions = rng.choice(
    len(X_representation),
    size=sample_size,
    replace=False,
)

isomap = Isomap(n_neighbors=20, n_components=2, n_jobs=-1)
isomap_projection = isomap.fit_transform(
    X_representation[sample_positions]
)

plt.figure(figsize=(8, 6))
points = plt.scatter(
    isomap_projection[:, 0],
    isomap_projection[:, 1],
    c=y_train.iloc[sample_positions],
    cmap="viridis",
    s=12,
    alpha=0.7,
)
plt.colorbar(points, label="Advertised price")
plt.xlabel("Isomap component 1")
plt.ylabel("Isomap component 2")
plt.title("Isomap projection coloured by price")
plt.tight_layout()
plt.show()


This representation uses supervised target encoding because it examines
the feature space seen by the predictive models. It should not be
interpreted as a fully unsupervised view of the raw data.


## 9. Polynomial numerical baseline


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

polynomial_model = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0)),
    ]
)
polynomial_model.fit(X_train[NUMERICAL], y_train)
polynomial_metrics = evaluate_regressor(
    polynomial_model,
    X_test[NUMERICAL],
    y_test,
)
polynomial_metrics


The original degree-two polynomial model reached a test R² of 0.388.
This supports the conclusion that numerical mileage and year alone do
not capture the high-dimensional categorical structure of vehicle
pricing.


## 10. Target-independent exploratory clustering


In [ ]:
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import TruncatedSVD

cluster_sample = X_train.sample(
    min(10_000, len(X_train)),
    random_state=RANDOM_STATE,
)
cluster_preprocessor = make_unsupervised_preprocessor()
cluster_features = cluster_preprocessor.fit_transform(cluster_sample)

inertias = []
k_values = range(2, 11)
for k in k_values:
    model = MiniBatchKMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        n_init=10,
        batch_size=1_024,
    )
    model.fit(cluster_features)
    inertias.append(model.inertia_)

plt.figure(figsize=(7, 4.5))
plt.plot(k_values, inertias, marker="o")
plt.xlabel("Number of clusters")
plt.ylabel("Inertia")
plt.title("Elbow analysis")
plt.tight_layout()
plt.show()


In [ ]:
kmeans = MiniBatchKMeans(
    n_clusters=5,
    random_state=RANDOM_STATE,
    n_init=10,
    batch_size=1_024,
)
labels = kmeans.fit_predict(cluster_features)

projection = TruncatedSVD(
    n_components=2,
    random_state=RANDOM_STATE,
).fit_transform(cluster_features)

plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=projection[:, 0],
    y=projection[:, 1],
    hue=labels.astype(str),
    palette="Set2",
    s=18,
    linewidth=0,
)
plt.xlabel("SVD component 1")
plt.ylabel("SVD component 2")
plt.title("Exploratory vehicle segments (k=5)")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()


The clustering representation deliberately avoids target encoding, so
price does not define the clusters. The labels are exploratory and are
not claimed to improve prediction unless a downstream cross-validated
comparison demonstrates that benefit.


## 11. Conclusion

The advanced stage shows that tree ensembles capture the non-linear and
high-cardinality structure more effectively than the classical linear
baseline. The original experiment recorded the best hold-out result for
stacking (R² 0.862, MAE 1,839), with XGBoost close behind and a smaller
generalisation gap than Random Forest. Explainability methods made the
influence of make, registration information, body type and mileage
visible at both global and local levels.

Future work should add repeated cross-validation, uncertainty intervals,
time-aware validation and a model card before any production use.
